# 🎓 Fine-tuning de Gemma 4 para que hable como Peñalara

**Curso práctico · ~75 minutos · Google Colab (GPU) · QLoRA con Unsloth**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_finetuning_gemma_seguros.ipynb)

En el [cuaderno anterior](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_selfhosted_pgvector.ipynb) montamos un RAG **100% self-hosted** que respondía con **Gemma 4 tal cual** (instruct). Funcionaba. Entonces… **¿para qué fine-tunear?**

> 🎯 **La respuesta honesta:** el fine-tuning **no le añade conocimiento** a Gemma (el conocimiento se lo da el RAG en el contexto). Lo que le da es **estilo y consistencia**: que **siempre** cite la cláusula, que **nunca** diga "cubierto" cuando el fragmento es una exclusión, que use el tono de un asesor de Peñalara y responda con el formato que la empresa quiere. Un modelo instruct hace *casi siempre* lo correcto; un modelo afinado lo hace **de forma fiable**.

Al terminar tendrás un **adapter** (unos pocos MB) que enchufarás al RAG del cuaderno anterior.

> ⚠️ **Necesitas GPU** (*Entorno de ejecución → GPU*; T4 vale — el entrenamiento de Gemma 4 E4B en 4-bit entra en ~8 GB).


## 🎒 Kit de conceptos (fine-tuning en 2 minutos)

- **Fine-tuning** — seguir entrenando un modelo ya hecho con **tus** ejemplos, para que adopte un comportamiento concreto.
- **LoRA** — en vez de mover los **miles de millones** de pesos del modelo (caro), se entrenan unas **matrices pequeñas** que se "suman" a las capas. Resultado: un **adapter** minúsculo (MB) en lugar de un modelo entero (GB).
- **QLoRA** — LoRA **sobre el modelo cuantizado en 4-bit**. Es lo que permite entrenar un modelo de miles de millones de parámetros en una sola T4.
- **Adapter** — el ficherito resultante. Se **enchufa** encima del modelo base cuando quieres su comportamiento; sin él, el base sigue intacto.
- **Unsloth** — una librería que hace el QLoRA de Gemma **2× más rápido y con menos memoria**.

> 🧠 **Idea clave:** entrenamos **el estilo de responder**, no los datos de las pólizas. Por eso los ejemplos serán `(contexto recuperado + pregunta) → respuesta ideal`. El modelo aprende *cómo* usar el contexto, no *qué* pólizas existen.


---
# 0 · Setup (Unsloth) ⏱️ ~8 min

▶️ **Qué hace esta celda:** instala Unsloth y fija **`transformers==5.5.0`** (la versión que Unsloth pinea para Gemma 4). La instalación es la parte más delicada del cuaderno: usa `--no-deps` para no romper el PyTorch preinstalado de Colab.

> ⚠️ Si ves un error de versiones al cargar el modelo, reinicia el entorno (*Entorno de ejecución → Reiniciar*) y reejecuta. Si algo cambió en Unsloth, la celda de instalación oficial siempre está en su notebook `Gemma4_(E4B)-Text.ipynb`.

In [ ]:
%%capture
import os, torch
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # xformers a juego con el torch de Colab
    v = ".".join(str(torch.__version__).split(".")[:2])          # p.ej. "2.10"
    xformers = "xformers==" + {"2.10":"0.0.34","2.9":"0.0.33.post1","2.8":"0.0.32.post2"}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
# Gemma 4 exige exactamente esta versión de transformers
!pip install --no-deps "transformers==5.5.0" "tokenizers>=0.22.0,<=0.23.0"

▶️ **Qué hace esta celda:** comprueba la GPU.

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ Activa la GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU"
print("✅ GPU:", torch.cuda.get_device_name(0))

---
# 1 · El dataset sintético ⏱️ ~10 min

Como en los cursos de RAG, generamos los datos nosotros. Aquí cada ejemplo es una conversación **`(contexto + pregunta) → respuesta ideal`**, construida a partir del catálogo de pólizas. La respuesta ideal **siempre** cita la cláusula y distingue cobertura de exclusión: ese es el comportamiento que queremos grabar en el modelo.

▶️ **Qué hace esta celda:** el catálogo de pólizas (el mismo de los cursos de RAG). Solo los datos — aquí no generamos PDFs.

In [ ]:
import os

POLIZAS = [
    dict(
        path="HOGAR_PLUS.pdf", producto="Hogar Plus", codigo="HP-2026-01", version="3.2",
        coberturas=[
            ["Incendio, rayo y explosión", "300.000 €", "Sin franquicia"],
            ["Daños por agua por rotura accidental de conducciones", "50.000 €", "150 €"],
            ["Robo y expoliación en el interior de la vivienda", "30.000 €", "150 €"],
            ["Rotura de cristales y vitrocerámica", "3.000 €", "Sin franquicia"],
            ["Responsabilidad civil familiar", "150.000 €", "300 €"],
            ["Fenómenos atmosféricos (viento, pedrisco, nieve)", "100.000 €", "300 €"],
        ],
        exclusiones=[
            ("Daños por agua no cubiertos", [
                "Los daños causados por <b>humedades, condensación o filtraciones</b> a través de muros, "
                "fachadas, terrazas o cubiertas, aun cuando sean consecuencia de lluvia, nieve o granizo.",
                "Los daños derivados de <b>falta de mantenimiento</b> de las conducciones, así como la "
                "corrosión, el óxido o el desgaste paulatino de tuberías.",
                "El coste de <b>localización y reparación de la avería</b> cuando no se haya producido "
                "daño material indemnizable.",
                "Los daños por <b>agua de lluvia que penetre por ventanas, puertas o huecos dejados "
                "abiertos</b> o defectuosamente cerrados por el Asegurado.",
            ]),
            ("Exclusiones generales", [
                "Los daños causados con dolo o culpa grave del Asegurado.",
                "Los daños derivados de <b>vicio propio o defecto de construcción</b> preexistente.",
                "Los daños calificados como catástrofe nacional o cubiertos por el <b>Consorcio de "
                "Compensación de Seguros</b>.",
                "Los daños en <b>viviendas deshabitadas</b> más de 60 días consecutivos.",
            ]),
        ],
        franquicias=[["Vivienda habitual", "150 €", "300 € en RC familiar"],
                     ["Segunda residencia", "300 €", "600 € en daños por agua"],
                     ["Vivienda en alquiler", "300 €", "600 € en robo"]],
    ),
    dict(
        path="AUTO_TODO_RIESGO.pdf", producto="Auto Todo Riesgo", codigo="AT-2026-04", version="2.1",
        coberturas=[
            ["Responsabilidad civil obligatoria", "Ilimitada (legal)", "Sin franquicia"],
            ["Daños propios por colisión o vuelco", "Valor venal + 20%", "300 €"],
            ["Robo total o parcial del vehículo", "Valor venal", "300 €"],
            ["Incendio del vehículo", "Valor venal", "Sin franquicia"],
            ["Lunas (parabrisas, laterales y trasera)", "Sin límite", "Sin franquicia"],
            ["Asistencia en viaje desde kilómetro 0", "Incluida", "Sin franquicia"],
        ],
        exclusiones=[
            ("Circunstancias del conductor", [
                "Siniestros conduciendo bajo <b>influencia de bebidas alcohólicas</b>, drogas o estupefacientes.",
                "Siniestros cuando el conductor <b>carezca de permiso de conducción</b> en vigor.",
                "Siniestros en <b>carreras, apuestas o pruebas deportivas</b> y sus entrenamientos.",
            ]),
            ("Uso del vehículo", [
                "El uso como <b>autoescuela, alquiler sin conductor, taxi o VTC</b>, salvo declaración expresa.",
                "El transporte de <b>mercancías peligrosas</b> o de más ocupantes de los autorizados.",
                "Los daños circulando por <b>vías no aptas</b> para la circulación o fuera de calzada.",
            ]),
            ("Daños no indemnizables", [
                "El <b>desgaste, uso o defecto de conservación</b> de las piezas.",
                "Los daños <b>exclusivamente estéticos</b> que no afecten a la seguridad.",
                "La <b>depreciación</b> del vehículo tras la reparación.",
            ]),
        ],
        franquicias=[["Conductor > 25 años y > 2 años de carné", "300 €", "Sin franquicia en lunas"],
                     ["Conductor novel (< 2 años de carné)", "600 €", "600 € en daños propios"],
                     ["Conductor ocasional no declarado", "900 €", "900 € en daños propios"]],
    ),
    dict(
        path="SALUD_FAMILIAR.pdf", producto="Salud Familiar", codigo="SF-2026-02", version="1.4",
        coberturas=[
            ["Medicina primaria y especialidades", "Sin límite", "Sin franquicia"],
            ["Pruebas diagnósticas (analítica, radiología)", "Sin límite", "Sin franquicia"],
            ["Hospitalización y cirugía en centros concertados", "Sin límite", "Sin franquicia"],
            ["Urgencias 24 h en cuadro médico", "Sin límite", "Sin franquicia"],
            ["Fisioterapia y rehabilitación", "30 sesiones/año", "10 € por sesión"],
            ["Psicología clínica", "20 sesiones/año", "15 € por sesión"],
        ],
        exclusiones=[
            ("Periodos de carencia", [
                "Las <b>intervenciones quirúrgicas</b> tienen una carencia de <b>seis (6) meses</b>.",
                "El <b>parto y la asistencia al embarazo</b> tienen una carencia de <b>diez (10) meses</b>.",
                "Los <b>tratamientos de reproducción asistida</b> tienen una carencia de <b>veinticuatro "
                "(24) meses</b> y se limitan a tres ciclos.",
            ]),
            ("Prestaciones no cubiertas", [
                "Las <b>enfermedades preexistentes</b> no declaradas en el cuestionario de salud.",
                "La <b>cirugía estética</b> y todo tratamiento sin finalidad terapéutica.",
                "Los tratamientos de <b>odontología</b> salvo extracción y limpieza anual.",
                "Los <b>medicamentos y prótesis</b> no incluidos en el catálogo.",
                "La asistencia <b>fuera del cuadro médico</b>, salvo urgencia vital acreditada.",
            ]),
        ],
        franquicias=[["Modalidad sin copago", "Sin franquicia", "Sin franquicia"],
                     ["Modalidad con copago", "Según acto médico", "10 € consulta / 25 € urgencia"],
                     ["Modalidad reembolso", "20% del gasto", "Límite 60.000 €/año"]],
    ),
]

▶️ **Qué hace esta celda:** el generador. Por cada cláusula (cobertura o exclusión) crea varias preguntas con distinto fraseo, monta un contexto con la cláusula relevante + un par de distractoras (para enseñar a discriminar) y escribe la respuesta ideal en el estilo Peñalara. Añade casos "fuera de contexto" para enseñar a **no alucinar**.

In [ ]:
import re, random, json
random.seed(42)

INSTR = ("Eres el asistente de Peñalara Seguros. Responde SOLO con el CONTEXTO. "
         "Si el fragmento es de EXCLUSIONES, NO está cubierto. Cita póliza y cláusula. Si no consta, dilo.")
_limpiar = lambda t: re.sub(r"<[^>]+>", "", t).strip().rstrip(".")
Q_COB = ["¿Me cubre {t}?", "Tengo un problema con {t}, ¿está cubierto?", "¿Entra {t} en mi póliza?",
         "¿Me pagáis por {t}?", "Quería saber si {t} está incluido."]
Q_EXC = ["¿Me cubre {t}?", "¿Está cubierto {t}?", "Me ha pasado esto: {t}. ¿Me lo cubrís?", "¿Entra {t} en la póliza?"]

def _clausulas(p):
    out = []
    for i, (gar, lim, fr) in enumerate(p["coberturas"], 1):
        out.append(("3. COBERTURAS", f"{gar}: cubierto hasta {lim}, franquicia {fr}.",
                    "SI", f"{p['producto']}, cláusula 3.{i}", gar))
    for i, (tit, items) in enumerate(p["exclusiones"], 1):
        for j, it in enumerate(items):
            out.append((f"4. EXCLUSIONES > 4.{i} {tit}", _limpiar(it),
                        "NO", f"{p['producto']}, cláusula 4.{i}.{'abcdefg'[j]}", _limpiar(it)))
    return out

def _ejemplos(p):
    cls, res = _clausulas(p), []
    for (sec, txt, ver, cita, tema) in cls:
        tema_c = (tema[:52] + "…") if len(tema) > 54 else tema
        for pl in (Q_COB if ver == "SI" else Q_EXC):
            dist = random.sample([c for c in cls if c[3] != cita], k=min(2, len(cls)-1))
            ctx = [(sec, txt)] + [(d[0], d[1]) for d in dist]; random.shuffle(ctx)
            contexto = "\n\n".join(f"[Póliza: {p['producto']} | Sección: {s}]\n{t}" for s, t in ctx)
            resp = (f"Sí, {tema_c.lower()} está cubierto en tu póliza {p['producto']} ({cita})." if ver == "SI"
                    else f"No, «{tema_c}» figura como exclusión en tu póliza {p['producto']}, no está cubierto ({cita}).")
            res.append({"messages": [
                {"role": "user", "content": f"{INSTR}\n\n### CONTEXTO:\n{contexto}\n\n### PREGUNTA:\n" + pl.format(t=tema_c.lower())},
                {"role": "assistant", "content": resp}]})
    return res

data = [e for p in POLIZAS for e in _ejemplos(p)]
for q in ["¿Cubre un ataque de dragones?", "¿Me pagáis un viaje a Marte?", "¿Cubre daños por meteorito?"]:
    data.append({"messages": [
        {"role": "user", "content": f"{INSTR}\n\n### CONTEXTO:\n[Póliza: Hogar Plus | Sección: 3. COBERTURAS]\nIncendio: cubierto hasta 300.000 €.\n\n### PREGUNTA:\n{q}"},
        {"role": "assistant", "content": "No consta en tu póliza ninguna cobertura para ese supuesto."}]})
random.shuffle(data)
n_val = max(8, len(data)//10)
val, train = data[:n_val], data[n_val:]
print(f"✅ {len(data)} ejemplos · train {len(train)} · val {len(val)}")
print("\n--- ejemplo ---")
print(train[0]["messages"][0]["content"][:240], "...")
print("→", train[0]["messages"][1]["content"])

> 🚩 **CHECKPOINT 1** — ~190 ejemplos, con su reparto train/val. Fíjate en el ejemplo: la respuesta cita la cláusula y dice claramente sí/no. Eso es lo que el modelo va a interiorizar.

---
# 2 · Cargar Gemma 4 E4B con Unsloth ⏱️ ~5 min

▶️ **Qué hace esta celda:** carga el modelo base en 4-bit. Usamos **`FastModel`** (no `FastLanguageModel`) porque Gemma 4 E4B es multimodal; `FastModel` es el cargador unificado.

In [ ]:
from unsloth import FastModel

MAX_SEQ_LEN = 2048
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it",   # cuant dinámica 4-bit de Unsloth
    dtype = None,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,        # QLoRA: base en 4 bits
    full_finetuning = False,
)
print("✅ Gemma 4 E4B cargado en 4-bit")

▶️ **Qué hace esta celda:** guarda la respuesta del modelo **ANTES** de entrenar, a una pregunta trampa, para poder comparar el efecto del fine-tuning al final. Usamos un contexto fijo (simulando lo que devolvería el RAG).

> 🧠 En Gemma 4 el `tokenizer` es en realidad un *processor* multimodal; para decodificar tokens usamos su tokenizer interno (`tokenizer.tokenizer`).

In [ ]:
CONTEXTO_DEMO = (
    "[Póliza: Hogar Plus | Sección: 3. COBERTURAS]\n"
    "Daños por agua por rotura accidental de conducciones: cubierto hasta 50.000 €, franquicia 150 €.\n\n"
    "[Póliza: Hogar Plus | Sección: 4. EXCLUSIONES > 4.1 Daños por agua no cubiertos]\n"
    "Los daños por filtraciones a través de terrazas, aun cuando sean consecuencia de lluvia.")
PREGUNTA_DEMO = "Ha entrado agua de lluvia por la terraza y se me ha estropeado el parqué. ¿Me lo cubre?"

def generar(pregunta, contexto=CONTEXTO_DEMO, max_new_tokens=256):
    msgs = [{"role": "user", "content": f"{INSTR}\n\n### CONTEXTO:\n{contexto}\n\n### PREGUNTA:\n{pregunta}"}]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    tok = getattr(tokenizer, "tokenizer", tokenizer)
    return tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

respuesta_base = generar(PREGUNTA_DEMO)
print("🔵 ANTES de entrenar (Gemma base):\n", respuesta_base)

---
# 3 · Añadir los adaptadores LoRA ⏱️ ~2 min

▶️ **Qué hace esta celda:** envuelve el modelo con LoRA. `finetune_vision_layers=False` **apaga la torre de visión** (no la necesitamos y ahorra VRAM); solo entrenamos las capas de lenguaje. `r` y `lora_alpha` controlan el "tamaño" del adapter.

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,   # 👈 solo texto
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8, lora_alpha = 8, lora_dropout = 0, bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Adaptadores LoRA añadidos")

---
# 4 · Preparar el dataset para el entrenador ⏱️ ~3 min

▶️ **Qué hace esta celda:** aplica el **chat template de Gemma 4** a cada conversación y la convierte en texto. Cada ejemplo queda con los marcadores `<start_of_turn>user … <start_of_turn>model …` que el entrenador entiende.

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

def a_dataset(filas):
    textos = [tokenizer.apply_chat_template(r["messages"], tokenize=False,
                                            add_generation_prompt=False).removeprefix("<bos>")
              for r in filas]
    return Dataset.from_list([{"text": t} for t in textos])

ds_train, ds_val = a_dataset(train), a_dataset(val)
print("✅ Datasets listos ·", len(ds_train), "train /", len(ds_val), "val")
print("\n--- una conversación formateada (recorte) ---\n", ds_train[0]["text"][:400])

---
# 5 · Entrenar ⏱️ ~15-20 min

▶️ **Qué hace esta celda:** configura el entrenador. Con `train_on_responses_only` le decimos que **solo aprenda de la respuesta del asistente** (no del contexto+pregunta) — así el modelo aprende *a responder*, no a repetir el enunciado.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer,
    train_dataset = ds_train, eval_dataset = ds_val,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LEN,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,      # batch efectivo = 4
        warmup_steps = 5,
        num_train_epochs = 3,                 # dataset pequeño: 2-3 épocas
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# entrenar SOLO sobre la parte del assistant (marcadores de Gemma)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part    = "<start_of_turn>model\n",
)
print("✅ Entrenador listo. Si la celda siguiente falla en train_on_responses_only, prueba sin argumentos.")

▶️ **Qué hace esta celda:** ¡entrena! Verás la *loss* bajar. En T4 son unos 15-20 min para ~190 ejemplos × 3 épocas. ☕☕

In [ ]:
stats = trainer.train()
print("\n✅ Entrenamiento terminado")

---
# 6 · El antes y el después ⏱️ ~5 min

▶️ **Qué hace esta celda:** vuelve a hacer la **misma pregunta trampa** con el modelo ya afinado, y la compara con la respuesta de antes. Aquí se ve —o no— el efecto del fine-tuning.

In [ ]:
FastModel.for_inference(model)
respuesta_ft = generar(PREGUNTA_DEMO)

print("PREGUNTA:", PREGUNTA_DEMO, "\n")
print("🔵 ANTES (Gemma base):\n", respuesta_base, "\n")
print("🟢 DESPUÉS (fine-tuned Peñalara):\n", respuesta_ft)

> 🎓 **Qué mirar.** Ambos deberían decir "no cubierto" (el RAG ya daba el contexto correcto). Pero el afinado debería hacerlo **con el estilo Peñalara**: más directo, citando la cláusula `4.1`, sin rodeos. El fine-tuning no cambió *lo que sabe*, cambió *cómo lo dice* — y esa fiabilidad de formato es justo lo que una empresa necesita para poner esto delante de un cliente.

▶️ **Qué hace esta celda:** guarda **solo el adapter** (unos MB). *No* fusionamos con el base (el merge de Gemma 4 tiene un bug conocido y, además, el adapter suelto es justo lo que queremos para enchufarlo al RAG).

In [ ]:
ADAPTER_DIR = "gemma4_penalara_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
import os
print("✅ Adapter guardado en", ADAPTER_DIR)
print("   tamaño:", sum(os.path.getsize(os.path.join(ADAPTER_DIR, f))
                        for f in os.listdir(ADAPTER_DIR))//1024//1024, "MB")

# (opcional) guardarlo en tu Drive para no perderlo al cerrar el Colab:
# from google.colab import drive; drive.mount("/content/drive")
# import shutil; shutil.copytree(ADAPTER_DIR, "/content/drive/MyDrive/"+ADAPTER_DIR, dirs_exist_ok=True)

---
# 7 · Enchufar el adapter al RAG ⏱️ ~5 min

El adapter es formato **PEFT estándar**, así que se carga **sin Unsloth** — con `transformers` + `peft` — encima del mismo `google/gemma-4-E4B-it` del cuaderno anterior. Este es el snippet que sustituiría la carga de Gemma en el **[Cuaderno A](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_selfhosted_pgvector.ipynb)** (bloque 3):

```python
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

GEMMA_ID = "google/gemma-4-E4B-it"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

base = AutoModelForCausalLM.from_pretrained(GEMMA_ID, quantization_config=bnb,
                                            device_map="auto", attn_implementation="eager")
gemma = PeftModel.from_pretrained(base, "gemma4_penalara_adapter")   # 👈 el adapter encima
gemma_tok = AutoTokenizer.from_pretrained(GEMMA_ID)
# ...y el resto del RAG (buscar_clausulas, responder) queda IGUAL.
```

> 🧠 **Por qué funciona sin Unsloth:** el adapter es un puñado de matrices LoRA con los nombres de módulo estándar (`q_proj`, `v_proj`…). `PeftModel` las coloca sobre el base. Le pasas el base explícito (`google/gemma-4-E4B-it`), así que da igual que el `adapter_config.json` mencione el repo de Unsloth.

> 🎓 **La lección de la serie completa:** cada pieza es **desacoplable**. Cambiaste el OCR, luego el vector store, luego los embeddings, luego el modelo generador — y ahora incluso **afinas ese generador** y lo vuelves a enchufar, sin tocar el resto del pipeline. Eso es una arquitectura sana.

---
# 8 · Cierre

### 🧪 Autoevaluación

1. El fine-tuning, ¿le enseñó a Gemma *qué* pólizas existen o *cómo* responder? ¿Por qué esa distinción importa en un RAG?
2. ¿Qué es un adapter LoRA y por qué ocupa MB y no GB?
3. ¿Por qué guardamos solo el adapter y no el modelo fusionado?
4. ¿Por qué el adapter se puede cargar con `peft` sin Unsloth?
5. ¿Qué hace `train_on_responses_only` y por qué es importante?

### Ejercicios para casa

- **Fácil** — Añade a `Q_COB`/`Q_EXC` fraseos más coloquiales y reentrena. ¿Generaliza mejor?
- **Medio** — Amplía el dataset con **preguntas trampa** (mismo riesgo en cobertura y exclusión) y mide si el afinado acierta más que el base.
- **Medio** — Sube `r` a 16 y compara calidad vs. tamaño del adapter y tiempo.
- **Difícil** — Enchufa este adapter al RAG del cuaderno A y corre su mini-evaluación (los `casos`): ¿mejora el % de aciertos frente a Gemma base?
- **Difícil** — Genera el dataset con un LLM (self-instruct) en vez de con plantillas, y compara la variedad y la calidad del modelo resultante.

### 🎓 Cuándo fine-tunear (y cuándo no)

- **NO** hace falta si un buen prompt + RAG ya te da respuestas correctas y el formato te da igual.
- **SÍ** compensa cuando necesitas **consistencia de formato/tono/citación** a escala (atención al cliente, informes), cuando quieres **acortar el prompt** (el estilo ya está "dentro" del modelo) o cuando el dominio tiene jerga muy propia.
- Recuerda: el fine-tuning **no sustituye al RAG** para el conocimiento — lo **complementa** para el comportamiento.

### 📚 Para seguir
- [Guía Gemma 4 en Unsloth](https://unsloth.ai/docs/models/gemma-4/train) · [LoRA/QLoRA (paper)](https://arxiv.org/abs/2305.14314)
- **La serie completa**: [emails](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_emails_bigquery_v2.ipynb) · [PDF Document AI](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_pdf_polizas_bigquery.ipynb) · [PDF OCR OSS](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_pdf_ocr_opensource.ipynb) · [RAG self-hosted](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_selfhosted_pgvector.ipynb) · **este**
